In [ ]:
### Imports and settings

from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt

from usv_playpen.os_utils import configure_path
from usv_playpen.plot_style import apply_plot_style
from usv_playpen.visualizations.auxiliary_plot_functions import create_colormap
from usv_playpen.visualizations.make_usv_spectrograms import USVSpectrogramPlotter
from usv_playpen.visualizations.make_usv_spectrograms import plot_usv_property_histograms
from usv_playpen.visualizations.make_usv_spectrograms import plot_session_type_usv_counts
from usv_playpen.visualizations.make_usv_spectrograms import plot_session_usv_timeline
from usv_playpen.visualizations.make_usv_spectrograms import plot_umap_with_category_thumbnails

apply_plot_style()

base_path = Path.cwd().parent
with open(base_path / "_parameter_settings" / "visualizations_settings.json", 'r') as vis_settings_file:
    vis_settings = json.load(vis_settings_file)

male_color = vis_settings["male_colors"][0]
female_color = vis_settings["female_colors"][0]

female_cmap = create_colormap(input_parameter_dict={
    "cm_length": 255,
    "cm_name": "female_cm",
    "cm_type": "sequential",
    "cm_start": (
        int(female_color[1:3], 16),
        int(female_color[3:5], 16),
        int(female_color[5:7], 16),
    ),
    "cm_end": (255, 255, 255),
    "equalize_luminance": True,
    "match_luminance_by": "max",
    "change_saturation": 0.5,
    "cm_opacity": 1,
})

male_cmap = create_colormap(input_parameter_dict={
    "cm_length": 255,
    "cm_name": "male_cm",
    "cm_type": "sequential",
    "cm_start": (
        int(male_color[1:3], 16),
        int(male_color[3:5], 16),
        int(male_color[5:7], 16),
    ),
    "cm_end": (255, 255, 255),
    "equalize_luminance": True,
    "match_luminance_by": "max",
    "change_saturation": 0.5,
    "cm_opacity": 1,
})

## How to run

This notebook drives `USVSpectrogramPlotter` from `usv_playpen.visualizations.make_usv_spectrograms`, which renders USV spectrograms in three modes:

- `"single"` — one channel's spectrogram. Optionally stacks the raw waveform of that channel above. dB amplitude scale.
- `"all"` — every channel's spectrogram in a single vertically stacked figure (with optional raw-waveform row per channel). dB amplitude scale.
- `"stitched"` — session-timeline spectrogram built by placing the pre-computed `[0, 1]`-normalized per-USV spectrograms from the consolidated HDF5 store at their on-session start times. Gaps between USVs are exactly zero. Renders in **linear normalized amplitude** with a fixed `[0, 1]` colorbar (`cbar_limits` is ignored for this mode). Requires `cfg["consolidated_spectrograms_h5"]` to point at the store; freq axis is fixed by the store (~30–120 kHz linear, 128 bins), and `freq_limits` is honored by slicing rows.

### Steps

1. **Point `session_root` at a session directory** that contains a `*_int16.mmap*` concatenated multi-channel audio file (and, for `"stitched"`, a `*_usv_summary.csv` whose rows are 1:1 with the per-session entries in the consolidated h5).
2. **Override any defaults you care about** in the `make_usv_spectrograms` block of `vis_settings`. The most common knobs:
   - `mode` — `"single"` / `"all"` / `"stitched"`.
   - `channel_of_interest` — int (only used in `"single"` mode).
   - `plot_raw_audio` — bool. Used by `"single"` / `"all"`; ignored by `"stitched"`. Raw-waveform y-limits are auto-scaled to the window's actual amplitude range.
   - `time_window` — analysis window in seconds as `[start, end]`. An `end` of `0` is interpreted as "end of recording".
   - `freq_limits` — frequency display limits in **kHz** as `[lower, upper]`. For `"stitched"`, anything outside the store's `[~30, ~120]` kHz freq axis can't be rendered.
   - `consolidated_spectrograms_h5` — path to the stitched-mode source store.
   - `nfft`, `spectrogram_cmap`, `cbar_limits` (`[vmin, vmax]` in dB, applies only to `"single"` / `"all"`; either entry can be `null` for auto), `plot_cbar`.
   - `save_dir`, `save_fig`, `fig_format`, `fig_dpi`, `fig_size`, `transparent_fig_bg`.
3. **Run the cell below.** It instantiates `USVSpectrogramPlotter` and calls `make_usv_spectrograms()`, which dispatches to the method named by `mode`. The figure is also displayed inline. If `save_fig` is `True`, the figure is written under `save_dir` (defaults to `<session_root>/audio` when `save_dir` is empty).

Call the individual methods directly (`plotter.plot_single_channel(channel=...)`, `plotter.plot_all_channels()`, `plotter.plot_stitched()`) if you need finer control than the `mode` dispatcher provides.

In [ ]:
### Run USV spectrogram plotter

session_root = '/mnt/falkner/Bartul/Data/20230124_192908'

# Pass `cmap_override=<Colormap>` to use the per-sex colormaps built above
# instead of the matplotlib name in vis_settings['make_usv_spectrograms']['spectrogram_cmap'].
# Set to None to fall back to the settings string.
cmap_override = female_cmap   # or female_cmap, or None

plotter = USVSpectrogramPlotter(
    root_directory=session_root,
    visualizations_parameter_dict=vis_settings,
    message_output=print,
    cmap_override=cmap_override,
)

fig = plotter.make_usv_spectrograms()
plt.show()

## Pooled USV property histograms

`plot_usv_property_histograms` is a module-level helper (not a method on `USVSpectrogramPlotter`) that summarizes per-USV properties across many sessions in a single figure.

### Inputs
- **`sessions_txt_path`** — a text file with one session root path per line (blank lines and lines starting with `#` are skipped). The function runs every path through `os_utils.configure_path`, so a file written with `/mnt/falkner/...` paths works fine on macOS (auto-mapped to `/Volumes/falkner/...`) and vice-versa on Linux.
- **`output_path`** *(optional)* — where to write the figure. Also `configure_path`'d.
- **`noise_col_id`** / **`noise_categories`** — rows in each session's `*_usv_summary.csv` whose `noise_col_id` value is in `noise_categories` are dropped before pooling. Defaults: `"vae_supercategory"` and `(0,)`.

### What it plots
Five histograms, in panel order: `duration` (ms), `mean_amplitude` (a.u.), `mean_freq_hz` (kHz), `freq_bandwidth_hz` (kHz), `spectral_entropy` (a.u.). Each panel uses 36 linearly-spaced bins over the FeatureZoo theoretical range, with `histtype='stepfilled'`, fill `#808080`, edge `#000000`, top/right spines hidden. Title reports the number of sessions successfully loaded and the total number of pooled vocalizations.

In [ ]:
### Run pooled USV property histograms

sessions_txt_path = '/mnt/falkner/Bartul/modeling/input_files/behavioral_courtship_intact_partners_sessions_list.txt'

fig_hist = plot_usv_property_histograms(
    sessions_txt_path=sessions_txt_path,
    output_path=None,         
    fig_format='svg',         
    noise_col_id='vae_supercategory',
    noise_categories=(0,),
)
plt.show()

## Mean USVs per session by session type

`plot_session_type_usv_counts` takes **three** session-list txt files — one each for male-female, female-female, and lone-male sessions — and renders a horizontal bar chart comparing the mean number of non-noise USVs per session, with SEM error bars on each bar.

### Inputs
- **`male_female_txt_path`** / **`female_female_txt_path`** / **`lone_male_txt_path`** — same format as `sessions_txt_path` above (one session root per line; `#` / blank lines skipped; every path run through `os_utils.configure_path`).
- **`output_path`**, **`fig_format`** — same save semantics as the histogram function.
- **`noise_col_id`** / **`noise_categories`** — used to drop noise rows before counting. Defaults `"vae_supercategory"` and `(0,)`.

### Colors
- Lone male       → `#9AC0CD` (project male color).
- Female-female   → `#FF6347` (project female color).
- Male-female     → `#DFB8B3` (pastel midpoint between male and female, mixed 35% toward white).

### Stats
For each session list the function discovers `*_usv_summary.csv`, drops noise rows, and counts the rest. SEM is `std(counts, ddof=1) / sqrt(n_sessions)`. Sessions whose CSV is missing or fails to read (SMB timeouts etc.) are logged and excluded from that group's mean.

In [ ]:
### Run mean USVs per session by session type

male_female_txt_path = '/mnt/falkner/Bartul/modeling/input_files/behavioral_courtship_intact_partners_sessions_list.txt'
female_female_txt_path = '/mnt/falkner/Bartul/modeling/input_files/behavioral_female_female_sessions_list.txt'  
lone_male_txt_path = '/mnt/falkner/Bartul/modeling/input_files/ephys_lone_male_sessions_list.txt'  

fig_counts = plot_session_type_usv_counts(
    male_female_txt_path=male_female_txt_path,
    female_female_txt_path=female_female_txt_path,
    lone_male_txt_path=lone_male_txt_path,
    output_path='/Users/bmimica/Downloads/session_type_usv_counts',
    fig_format='svg',
    noise_col_id='vae_supercategory',
    noise_categories=(0,),
)
plt.show()

## Per-session USV timeline

`plot_session_usv_timeline` renders every non-noise USV in one session as a colored rectangle spanning its `[start, stop]` interval on a single horizontal strip. Color encodes the emitter: male / female / unassigned.

### Inputs
- **`session_root`** — session directory path (`/mnt/...` or `/Volumes/...`, run through `configure_path` internally).
- **`time_window`** *(optional)* — `(start_s, end_s)` to clip the strip to a sub-window of the session; `None` shows the whole session.
- **`output_path`**, **`fig_format`** — same save semantics as the other functions.
- **`noise_col_id`** / **`noise_categories`** — drop noise rows before drawing. Defaults `"vae_supercategory"` and `(0,)`.

### How emitter → color
The session's male / female track ids are read from `<session>/video/*_points3d_translated_rotated_metric.h5` (`track_names[0]` = male, `track_names[1]` = female — same convention as `usv_summary_statistics.extract_session_metadata`). Each USV's CSV `emitter` field is matched against these ids:
- match male id → `#9AC0CD`
- match female id → `#FF6347`
- anything else → `#C0C0C0` (unassigned)

A 3-entry legend reports per-group counts.

In [ ]:
### Run per-session USV timeline

fig_timeline = plot_session_usv_timeline(
    session_root='/mnt/falkner/Bartul/Data/20230119_172410',
    time_window=[457, 462],                
    output_path='/Users/bmimica/Downloads/session_usv_timeline_20230119_172410_457s-462s', 
    fig_format='svg',       
    noise_col_id='vae_supercategory',
    noise_categories=(0,),
)
plt.show()

## UMAP + per-category spectrogram thumbnails

`plot_umap_with_category_thumbnails` renders a static two-panel figure:

- **Left**: UMAP scatter (VAE or QLVM) colored by category / supercategory.
- **Right**: a grid of spectrogram thumbnails — one row per category, each row a random sample of `n_samples_per_category` USVs from that category. Every row is outlined in the matching category color so it lines up visually with the scatter.

### Inputs
- `sessions_txt_path` — text file listing session roots.
- `consolidated_h5_path` — path to the SAM2 + spectrograms HDF5.
- `map_type` — `"vae"` or `"qlvm"`.
- `category_col_suffix` — `"category"` or `"supercategory"`.
- `n_samples_per_category` — thumbnails per row.
- `apply_mask` — global SAM2 mask toggle. Masks are pre-flipped vertically to match the spec's `origin='lower'` convention.
- `mask_excluded_categories` — tuple of category integer values that should be shown WITHOUT a mask even when `apply_mask=True`. Use this when SAM2 misses the call type and the masked view destroys the spectrogram.
- `category_colors` — optional `{int: hex}` mapping. Defaults to `tab10` / `tab20`.
- `scatter_max_points` — caps how many points are rendered in the scatter (per-category sampling for the thumbnails still draws from the full pool).
- `output_path`, `fig_format` — same save semantics as the other helpers.
- `pooled_df` — pass a pre-loaded pooled-embeddings DataFrame to skip re-reading CSVs.

In [ ]:
### Run UMAP + per-category spectrogram thumbnails

from usv_playpen.visualizations.make_usv_spectrograms import (
    plot_umap_with_category_thumbnails,
)

sessions_txt_path = '/mnt/falkner/Bartul/modeling/input_files/behavioral_courtship_intact_partners_sessions_list.txt'
consolidated_h5_path = '/Volumes/falkner/Bartul/spectrograms/spectrograms_sam2masks_553sessions_602724vocalizations_20260514_153009.h5'

# Small standalone h5 with QLVM cluster centers + sizes for both
# resolutions. Built by /tmp/build_qlvm_clusters_h5.py from the
# Dexter QLVM provenance JSONs. Picked up automatically based on
# `category_col_suffix`:
#   - 'supercategory' -> /coarse (7 centers, label 1..7)
#   - 'category'      -> /fine   (12 centers, label 1..12)
qlvm_clusters_h5 = '/Volumes/falkner/Bartul/spectrograms/qlvm_clusters_20260506.h5'

fig_umap_cats = plot_umap_with_category_thumbnails(
    sessions_txt_path=sessions_txt_path,
    consolidated_h5_path=consolidated_h5_path,
    map_type='qlvm',                      # 'vae' or 'qlvm'
    category_col_suffix='supercategory',  # 'category' (fine) or 'supercategory' (coarse)
    n_samples_per_category=10,
    apply_mask=True,
    mask_excluded_categories=(),
    category_colors=None,
    # `sampling_method` -- per-cluster pick strategy:
    #   - 'random'         : uniform random draw within the cluster.
    #   - 'nearest'        : the n_per points closest to the cluster
    #                        centroid (most "typical" samples).
    #   - 'farthest_point' : iterative farthest-point sampling from the
    #                        centroid; maximally spread coverage.
    #   - 'grid'           : overlay a grid on the cluster's bounding
    #                        box, snap each grid cell to its nearest
    #                        unused data point.
    #   - 'spiral'         : Archimedean spiral from the cluster center
    #                        (uses `cluster_centers_h5_path` when given;
    #                        otherwise falls back to the data centroid)
    #                        out to `r_max`, snap each spiral position
    #                        to its nearest unused data point.
    sampling_method='spiral',
    cluster_centers_h5_path=qlvm_clusters_h5,
    # --- spiral sampling + (optional) overlay ----------------------------------
    # The spiral STILL controls where the per-cluster picks come from
    # (radius_scale / radius_abs / n_turns shape the sampling
    # trajectory). `draw_spiral_overlay` toggles whether the spiral
    # line is also drawn on the scatter for inspection. Numbered picks
    # are drawn separately via `annotate_picks_on_scatter`.
    #
    # Sizing modes for the spiral sampling trajectory:
    #   - `spiral_radius_abs` (float, embedding-space units): same
    #     absolute radius for every cluster -> uniform.
    #   - otherwise `spiral_radius_scale` multiplies that cluster's
    #     own max-distance r_max (per-cluster proportional).
    #
    # Variation across runs: `spiral_random_phase=True` rotates each
    # cluster's spiral by a random starting angle so the picks land on
    # different rays each call. The actual random draws still go
    # through `seed` -- pass `seed=None` for OS-entropy randomness
    # (different specs every run), or pass an explicit int to
    # reproduce a particular figure.
    draw_spiral_overlay=True,
    spiral_show_only_for=None,
    spiral_color='#000000',
    spiral_linewidth=1.0,
    spiral_radius_scale=0.1,
    spiral_radius_abs=0.1,                # e.g. 2.0 -> uniform 2.0-unit spiral on every cluster
    spiral_n_turns=3,
    spiral_random_phase=True,             # random starting angle per cluster
    seed=None,                            # None -> different picks each run; pass an int to reproduce
    # ---------------------------------------------------------------------------
    draw_cluster_boundaries=True,
    knn_boundary_neighbors=15,
    knn_boundary_resolution=200,
    annotate_picks_on_scatter=True,
    pick_number_fontsize=9,              # fontsize for the numbered picks on the top scatter
    scatter_max_points=50_000,
    fig_size=(16, 12),
    fig_dpi=300,
    output_path=None,
    fig_format='svg',
    embeddings_cache_path='/Users/bmimica/.usv_playpen_cache/pooled_embeddings.parquet',
)
plt.show()